# Module 04 — Modular Arithmetic
### Mathematics of Cryptography · Python Companion

---

Modular arithmetic is the engine inside almost every modern cryptographic system. RSA keys are computed using modular exponentiation. Diffie-Hellman key exchange lives entirely in a multiplicative group modulo a prime. AES's finite-field arithmetic is a sophisticated cousin of modular thinking.

The idea is simple: instead of an infinite number line, you work on a circular track of size `n`. Every integer maps to a *landing position* between 0 and n−1. That position is the *remainder*.

**What you will build:**
- Functions that show full division-form working for any `a mod n`
- Congruence testers using both equivalent definitions
- Modular add, subtract, and multiply with worked output
- A Caesar cipher (encrypt and decrypt)
- A brute-force Caesar cracker using frequency analysis

**Prerequisites:** Basic Python, familiarity with integer division. No prior cryptography needed.

---
## Section 1 — Setup and Display Helpers

In [ ]:
def show_mod(a, n):
    """Show a mod n with full division-form working and a verification check."""
    q, r = divmod(a, n)   # Python's divmod: q = quotient, r = remainder
    print(f"  {a} ÷ {n}:")
    print(f"    quotient  q = {q}")
    print(f"    remainder r = {r}")
    print(f"    division form: {a} = {q}·{n} + {r}")
    print(f"    {a} mod {n} = {r}")
    assert a == q * n + r, "Division form check failed!"
    assert 0 <= r < n,     "Remainder out of range!"
    print(f"    ✓ verified: 0 ≤ {r} < {n}")
    return r

def mod_pos(a, n):
    """Always-positive modular reduction. Safe for negative a."""
    return ((a % n) + n) % n

def separator(title=""):
    line = "─" * 52
    if title:
        print(f"\n── {title} " + "─" * max(0, 48 - len(title)))
    else:
        print(line)

print("Helpers loaded.")

---
## Section 2 — The `%` Operator and `divmod()`

Python has two tools for modular reduction:
- `a % n` — returns just the remainder
- `divmod(a, n)` — returns `(quotient, remainder)` together

Both always return a non-negative remainder when `n > 0` — this is a Python design choice that differs from some other languages.

In [ ]:
# Reproduce the tutorial's example table
examples = [
    (14,  5),
    (21,  6),
    (32,  8),
    (255, 16),
    (17,  5),
    (41,  12),
]

separator("Mod operation table")
print(f"  {'Expression':>16}  {'Division form':^22}  {'Remainder':>9}")
print(f"  {'----------':>16}  {'------------':^22}  {'---------':>9}")
for a, n in examples:
    q, r = divmod(a, n)
    expr  = f"{a} mod {n}"
    div_f = f"{a} = {q}·{n} + {r}"
    print(f"  {expr:>16}  {div_f:^22}  {r:>9}")

In [ ]:
# Use show_mod() for the detailed view
separator("Detailed: 47 mod 12")
show_mod(47, 12)

separator("Detailed: 255 mod 16")
show_mod(255, 16)

### Exercise 2.1

Predict the remainder before running, then verify:
1. `73 mod 9`
2. `100 mod 7`
3. `256 mod 256`

In [ ]:
# Your predictions here (as comments), then run show_mod() to verify


---
## Section 3 — Python vs JavaScript: The Negative Modulo Gotcha

This is the single most important thing to understand when translating between Python and JavaScript code.

**Python:** `a % n` is always in `[0, n−1]` for positive `n`. Python floors toward negative infinity.

**JavaScript:** `-5 % 12 === -5`. JS truncates toward zero, so the result can be negative.

The JS-safe pattern is `((a % n) + n) % n` — which Python doesn't need but works in both languages.

In [ ]:
separator("Python % always returns non-negative for positive n")
for a in [-13, -5, -1, 0, 1, 5, 17]:
    r = a % 12
    safe = mod_pos(a, 12)
    match = "✓" if r == safe else "✗"
    print(f"  {a:>4} % 12 = {r:>2}   mod_pos = {safe:>2}   {match}")

print()
print("  Python's % and mod_pos() always agree for positive n.")
print("  In JavaScript you MUST write ((a % n) + n) % n for subtraction.")

In [ ]:
# Verify the two functions agree across a wide range
n = 12
mismatches = [(a, a%n, mod_pos(a,n)) for a in range(-100, 100) if a%n != mod_pos(a,n)]
if mismatches:
    print("MISMATCHES:", mismatches)
else:
    print("  No mismatches — Python % and mod_pos() agree for all a in [-100, 100).")

---
## Section 4 — Congruence

Two integers `a` and `b` are **congruent modulo `n`** if they have the same remainder. There are two equivalent ways to test this:

1. **Remainder test:** `a % n == b % n`
2. **Divisibility test:** `(a - b) % n == 0`

Both always agree. Which you use depends on context.

In [ ]:
def congruent_remainder(a, b, n):
    """Test a ≡ b (mod n) using the remainder definition."""
    ra, rb = a % n, b % n
    result = (ra == rb)
    print(f"  {a} mod {n} = {ra},  {b} mod {n} = {rb}  →  {'congruent' if result else 'NOT congruent'}")
    return result

def congruent_divisibility(a, b, n):
    """Test a ≡ b (mod n) using the divisibility definition."""
    diff = a - b
    result = (diff % n == 0)
    print(f"  {a} - {b} = {diff};  {diff} divisible by {n}? {'Yes' if result else 'No'}")
    return result

separator("17 ≡ 5 (mod 12) — should be True")
r1 = congruent_remainder(17, 5, 12)
r2 = congruent_divisibility(17, 5, 12)
print(f"  Both tests agree: {r1 == r2}")

separator("38 ≡ 2 (mod 12) — should be True")
congruent_remainder(38, 2, 12)
congruent_divisibility(38, 2, 12)

separator("29 ≡ 4 (mod 13) — should be False")
congruent_remainder(29, 4, 13)
congruent_divisibility(29, 4, 13)

In [ ]:
# Equivalence classes: show all integers from -5 to 35 grouped by their residue mod 7
separator("Z_7 equivalence classes — integers grouped by residue mod 7")
n = 7
classes = {r: [] for r in range(n)}
for a in range(-5, 36):
    classes[a % n].append(a)

for r in range(n):
    print(f"  Residue {r}: {classes[r]}")

print()
print("  Every integer lands on exactly one residue. Congruent integers share a class.")

### Exercise 4.1

Test each statement using both `congruent_remainder` and `congruent_divisibility`:
1. `44 ≡ 8 (mod 9)`  — predict YES or NO first
2. `101 ≡ 1 (mod 10)` — predict first
3. `29 ≡ 3 (mod 13)` — predict first

In [ ]:
# Your work here


---
## Section 5 — Modular Addition, Subtraction, and Multiplication

In [ ]:
def mod_add(a, b, n):
    raw = a + b
    r   = raw % n
    print(f"  ({a} + {b}) mod {n}:  {a} + {b} = {raw};  {raw} mod {n} = {r}")
    return r

def mod_sub(a, b, n):
    raw = a - b
    r   = mod_pos(raw, n)
    note = f"  (negative → add {n}: {raw} + {n} = {r})" if raw < 0 else ""
    print(f"  ({a} - {b}) mod {n}:  {a} - {b} = {raw}{note};  result = {r}")
    return r

def mod_mul(a, b, n):
    raw = a * b
    q, r = divmod(raw, n)
    print(f"  ({a} × {b}) mod {n}:  {a} × {b} = {raw} = {q}·{n} + {r};  result = {r}")
    return r

separator("Addition examples from the tutorial")
mod_add(19, 11, 26)   # tutorial example: C ≡ P + K (mod 26)
mod_add(7,  8,  12)   # clock arithmetic
mod_add(8,  25, 26)

separator("Subtraction — negative wrap")
mod_sub(3,  8,  12)   # tutorial example
mod_sub(4,  13, 26)

separator("Multiplication")
mod_mul(7,  9,  26)   # tutorial example
mod_mul(8,  7,  26)
mod_mul(15, 15, 26)

In [ ]:
# Print the full addition table for Z_7
separator("Addition table for Z_7")
n = 7
header = "  +  " + " ".join(f"{j:>3}" for j in range(n))
print(header)
print("  " + "─" * (len(header) - 2))
for i in range(n):
    row = f"  {i:>2} |" + " ".join(f"{(i+j)%n:>3}" for j in range(n))
    print(row)

print()
print("  Every row and column contains each residue exactly once.")
print("  This is a property of Z_n addition when n is any positive integer.")

In [ ]:
# Print the multiplication table for Z_7
separator("Multiplication table for Z_7")
n = 7
header = "  ×  " + " ".join(f"{j:>3}" for j in range(n))
print(header)
print("  " + "─" * (len(header) - 2))
for i in range(n):
    row = f"  {i:>2} |" + " ".join(f"{(i*j)%n:>3}" for j in range(n))
    print(row)

print()
print("  Note row 0: all zeros. Row 1: identity. Can you find every non-zero row contains 1?")
print("  If every non-zero element has a row containing 1, it has a multiplicative inverse.")
print("  (That is the subject of Module 05.)")

### Exercise 5.1

1. Compute `(200 + 56) mod 256` and `(255 + 1) mod 256`. What do you notice about the second result? How does this relate to byte overflow from Module 01?
2. Print the addition table for `Z_5`.
3. Compute `(7 × 9) mod 26` and `(8 × 7) mod 26` using `mod_mul()`.

In [ ]:
# Your work here


---
## Section 6 — Caesar Cipher

The Caesar cipher is modular addition applied to letters. Map A=0, B=1, …, Z=25. Shift by key `K`:

$$C \equiv P + K \pmod{26}$$

Decryption is modular subtraction: $P \equiv C - K \pmod{26}$

In [ ]:
def caesar_encrypt_letter(letter, k):
    """Encrypt a single letter with Caesar key k. Shows working."""
    p = ord(letter.upper()) - ord('A')
    c = (p + k) % 26
    C = chr(c + ord('A'))
    need_wrap = (p + k) >= 26
    note = f" → wraps: {p+k} mod 26 = {c}" if need_wrap else ""
    print(f"  {letter.upper()} = {p:>2},  {p} + {k} = {p+k}{note}  →  {C}")
    return C

def caesar_decrypt_letter(letter, k):
    """Decrypt a single letter with Caesar key k."""
    c = ord(letter.upper()) - ord('A')
    p = mod_pos(c - k, 26)   # Python handles negative correctly
    P = chr(p + ord('A'))
    note = f" → negative, add 26: {c-k}+26={p}" if (c - k) < 0 else ""
    print(f"  {letter.upper()} = {c:>2},  {c} - {k} = {c-k}{note}  →  {P}")
    return P

def caesar_encrypt(text, k):
    """Encrypt a full message, preserving spaces and punctuation."""
    return ''.join(
        chr((ord(ch) - ord('A') + k) % 26 + ord('A')) if ch.isalpha() else ch
        for ch in text.upper()
    )

def caesar_decrypt(text, k):
    """Decrypt a full message."""
    return caesar_encrypt(text, 26 - k)   # decrypting with k = encrypting with 26-k

separator("Tutorial examples: encrypt letters")
caesar_encrypt_letter('Y', 5)   # tutorial: Y shifted by 5 → D
caesar_encrypt_letter('A', 3)
caesar_encrypt_letter('Z', 2)

separator("Decrypt examples")
caesar_decrypt_letter('D', 5)   # should give Y back
caesar_decrypt_letter('A', 3)   # decrypt wraps below 0

In [ ]:
separator("Encrypt and decrypt a full message")
plaintext  = "MODULAR ARITHMETIC"
key        = 13
ciphertext = caesar_encrypt(plaintext, key)
recovered  = caesar_decrypt(ciphertext, key)

print(f"  Plaintext  : {plaintext}")
print(f"  Key        : {key}")
print(f"  Ciphertext : {ciphertext}")
print(f"  Decrypted  : {recovered}")
print(f"  Recovered? : {recovered == plaintext}")

In [ ]:
# ROT-13: key = 13 is self-inverse because 13 + 13 = 26 ≡ 0 (mod 26)
separator("ROT-13: the self-inverse cipher")
msg = "HELLO WORLD"
rot13_once  = caesar_encrypt(msg, 13)
rot13_twice = caesar_encrypt(rot13_once, 13)

print(f"  Original      : {msg}")
print(f"  After ROT-13  : {rot13_once}")
print(f"  After ROT-13² : {rot13_twice}")
print()
print("  ROT-13 applied twice recovers the original because 13 + 13 ≡ 0 (mod 26).")
print("  Encryption and decryption are the same operation — encryption IS decryption.")

---
## Section 7 — Breaking Caesar: Brute Force and Frequency Analysis

Caesar has only 26 possible keys. A brute-force attack simply tries all of them. The correct key produces readable English; the wrong keys produce gibberish.

Frequency analysis makes it even faster: the most common letter in English is E (about 13%). Whatever letter appears most in the ciphertext is likely E. That immediately suggests the key.

In [ ]:
def brute_force_caesar(ciphertext):
    """Try all 26 keys and print each candidate plaintext."""
    print(f"  Ciphertext: {ciphertext}")
    print()
    for k in range(26):
        candidate = caesar_decrypt(ciphertext, k)
        print(f"  K={k:>2}: {candidate}")

def frequency_attack(ciphertext):
    """Guess the key by matching the most frequent letter to E."""
    letters_only = [ch for ch in ciphertext.upper() if ch.isalpha()]
    if not letters_only:
        return None
    # Count frequencies
    counts = {}
    for ch in letters_only:
        counts[ch] = counts.get(ch, 0) + 1
    # Most frequent letter
    most_common = max(counts, key=counts.get)
    # If most_common is encrypted E, then key = (most_common - E) mod 26
    e_pos = 4   # E = 4
    mc_pos = ord(most_common) - ord('A')
    guessed_key = (mc_pos - e_pos) % 26

    print(f"  Most frequent letter in ciphertext: {most_common} (count: {counts[most_common]})")
    print(f"  If {most_common} = encrypted E, then key K = ({mc_pos} - 4) mod 26 = {guessed_key}")
    print(f"  Candidate plaintext: {caesar_decrypt(ciphertext, guessed_key)}")
    return guessed_key

In [ ]:
# The hidden message — try to crack it
mystery = "FTUE UE DZOARG MDUFXBTUMZ"

separator("Frequency analysis attack")
guessed = frequency_attack(mystery)

print()
separator("Brute force (all 26 keys)")
brute_force_caesar(mystery)

### Exercise 7.1

The ciphertext `"FTUE UE DZOARG MDUFXBTUMZ"` was encrypted with a single Caesar key. Use both attacks above to find the key and the original message.

1. What key does the frequency analysis guess?
2. Does brute force confirm it?
3. Encrypt your own short message with a key of your choice. Give the ciphertext to a classmate and ask them to crack it.

In [ ]:
# Encrypt your own message
my_message = "WRITE YOUR MESSAGE HERE"
my_key     = 7   # change this

my_ciphertext = caesar_encrypt(my_message, my_key)
print(f"Plaintext:  {my_message}")
print(f"Key:        {my_key}")
print(f"Ciphertext: {my_ciphertext}")

---
## Section 8 — Modular Arithmetic and Byte Overflow

You saw in Module 01 that adding 1 to a full byte (255) wraps to 0. This is exactly modular arithmetic with modulus 256.

In [ ]:
separator("Byte arithmetic IS modular arithmetic mod 256")

# Overflow examples from Module 01, now in modular notation
examples = [
    (200, 100),
    (255,   1),
    (255, 255),
    (128, 128),
]

print(f"  {'a':>4} + {'b':>4}  =  {'true sum':>9}  {'mod 256':>8}  {'hex result':>12}")
for a, b in examples:
    true_sum = a + b
    byte_sum = true_sum % 256
    print(f"  {a:>4} + {b:>4}  =  {true_sum:>9}  {byte_sum:>8}  0x{byte_sum:02X}")

print()
print("  The 8-bit mask (& 0xFF in Python/JS) is identical to % 256.")
print("  Fixed-width registers are circular number systems — Z_256, Z_65536, Z_2^32, etc.")

---
## Section 9 — Summary and Final Challenge

### What you have built

| Concept | Python |
|---------|--------|
| Remainder `a mod n` | `a % n` or `divmod(a, n)[1]` |
| Division form | `q, r = divmod(a, n)` |
| Congruence test | `a % n == b % n` or `(a-b) % n == 0` |
| Residue set `Z_n` | `list(range(n))` |
| Mod add | `(a + b) % n` |
| Mod subtract (safe) | `((a - b) % n + n) % n` |
| Mod multiply | `(a * b) % n` |
| Caesar encrypt | `(p + k) % 26` |
| Caesar decrypt | `(c - k) % 26` (Python safe; JS: `((c-k)%26+26)%26`) |

### Cryptographic significance

- **Classical ciphers:** Caesar is `Z_26` addition. The Vigenère cipher is Caesar with a repeating key.
- **RSA:** exponentiation modulo a product of two large primes (`n = p × q`). The security depends on the difficulty of factoring `n`.
- **Diffie-Hellman:** `g^a mod p` and `g^b mod p` where `p` is a large prime.
- **AES:** uses arithmetic in `GF(2^8)` — a finite field, which is a more structured version of modular arithmetic.

### Final Challenge — How Secure Is Caesar?

Caesar has 26 keys. A modern cipher like AES-128 has `2^128` keys — about `3.4 × 10^38`.

If a computer could try one billion (`10^9`) keys per second, how long would it take to brute-force AES-128?

In [ ]:
keys_aes128     = 2**128
keys_per_second = 10**9
seconds         = keys_aes128 // keys_per_second

minutes = seconds // 60
hours   = minutes // 60
days    = hours   // 24
years   = days    // 365

# Age of the universe: ~13.8 billion years
universe_age_years = 13_800_000_000

print(f"  AES-128 key space : 2^128 = {keys_aes128:,.0f}")
print(f"  Keys/second       : {keys_per_second:,.0f}")
print(f"  Time to exhaust   : {years:,.0f} years")
print(f"  Age of universe   : {universe_age_years:,.0f} years")
print(f"  Ratio             : {years // universe_age_years:,.0f}× the age of the universe")
print()
print("  Caesar: 26 keys → cracked in microseconds.")
print("  AES-128: 2^128 keys → computationally infeasible even for all computers on Earth.")

---
## What Comes Next

**Module 05 — Multiplicative Inverses Modulo `n`** asks: when can we *undo* multiplication in modular arithmetic?

Modular addition can always be undone with subtraction. But `7 × 4 ≡ 28 ≡ 2 (mod 26)` — does there exist a number `x` such that `7 × x ≡ 1 (mod 26)`? Sometimes yes, sometimes no. The Extended Euclidean Algorithm answers this question, and it is the foundation of RSA key generation.

---
*Mathematics of Cryptography — Python Companion Series*